In [1]:
from tensorflow.python.keras.models import load_model
model = load_model('model.h5')

In [2]:
from konlpy.tag import Okt
from tensorflow.keras.utils import get_file
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
import numpy as np
from nltk import FreqDist
from functools import reduce
import os
import re
import matplotlib.pyplot as plt

In [3]:
okt = Okt()

In [4]:
TRAIN_FILE1 = os.path.join("doing_data_train.txt")
TEST_FILE1 = os.path.join("doing_data_test.txt")

In [5]:
def read_data(dir):
    stories, questions, answers = [], [], [] # 각각 스토리, 질문, 답변을 저장할 예정
    story_temp = [] # 현재 시점의 스토리 임시 저장
    lines = open(dir, "rb")

    for line in lines:
        line = line.decode("utf-8") # b' 제거
        line = line.strip() # '\n' 제거
        idx, text = line.split(" ", 1) # 맨 앞에 있는 id number 분리
        # 여기까지는 모든 줄에 적용되는 전처리

        if int(idx) == 1:
            story_temp = []

        if "\t" in text: # 현재 읽는 줄이 질문 (tab) 답변 (tab)인 경우
            question, answer, _ = text.split("\t") # 질문과 답변을 각각 저장
            stories.append([x for x in story_temp if x]) # 지금까지의 누적 스토리를 스토리에 저장
            questions.append(question)
            answers.append(answer)

        else: # 현재 읽는 줄이 스토리인 경우
            story_temp.append(text) # 임시 저장

    lines.close()
    return stories, questions, answers

In [6]:
train_data1 = read_data(TRAIN_FILE1)
test_data1 = read_data(TEST_FILE1)

In [7]:
train_stories, train_questions, train_answers = read_data(TRAIN_FILE1)
test_stories, test_questions, test_answers = read_data(TEST_FILE1)

In [8]:
def tokenize(sent):
    return [x.strip() for x in re.split('(\W+)?', sent) if x.strip()]

In [9]:
def preprocess_data(train_data, test_data):
    counter = FreqDist()

    # 두 문장의 story를 하나의 문장으로 통합하는 함수
    flatten = lambda data: reduce(lambda x, y: x + y, data)

    # 각 샘플의 길이를 저장하는 리스트
    story_len = []
    question_len = []

    for stories, questions, answers in [train_data1, test_data1]:
        for story in stories:
            stories = tokenize(flatten(story)) # 스토리의 문장들을 펼친 후 토큰화
            story_len.append(len(stories)) # 각 story의 길이 저장
            for word in stories: # 단어 집합에 단어 추가
                counter[word] += 1
        for question in questions:
            question = tokenize(question)
            question_len.append(len(question))
            for word in question:
                counter[word] += 1
        for answer in answers:
            answer = tokenize(answer)
            for word in answer:
                counter[word] += 1

    # 단어 집합 생성
    word2idx = {word : (idx + 1) for idx, (word, _) in enumerate(counter.most_common())}
    idx2word = {idx : word for word, idx in word2idx.items()}

    # 가장 긴 샘플의 길이
    story_max_len = np.max(story_len)
    question_max_len = np.max(question_len)

    return word2idx, idx2word, story_max_len, question_max_len

In [10]:
word2idx, idx2word, story_max_len, question_max_len = preprocess_data(train_data1, test_data1)

C:\Users\shlee\anaconda3\envs\develop\lib\re.py:212: FutureWarning: split() requires a non-empty pattern match.
  return _compile(pattern, flags).split(string, maxsplit)


In [11]:
vocab_size = len(word2idx) + 1

In [12]:
def vectorize(data, word2idx, story_maxlen, question_maxlen):
    Xs, Xq, Y = [], [], []
    flatten = lambda data: reduce(lambda x, y: x + y, data)

    stories, questions, answers = data
    for story, question, answer in zip(stories, questions, answers):
        xs = [word2idx[w] for w in tokenize(flatten(story))]
        xq = [word2idx[w] for w in tokenize(question)]
        Xs.append(xs)
        Xq.append(xq)
        Y.append(word2idx[answer])
        # 스토리와 질문은 각각의 최대 길이로 패딩
        # 정답은 원-핫 인코딩
    return pad_sequences(Xs, maxlen=story_maxlen),\
           pad_sequences(Xq, maxlen=question_maxlen),\
           to_categorical(Y, num_classes=len(word2idx) + 1)

In [13]:
Xstrain1, Xqtrain1, Ytrain1 = vectorize(train_data1, word2idx, story_max_len, question_max_len)
Xstest1, Xqtest1, Ytest1 = vectorize(test_data1, word2idx, story_max_len, question_max_len)

In [16]:
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Embedding
from tensorflow.keras.layers import Permute, dot, add, concatenate
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input, Activation

In [1]:
# plot accuracy and loss plot
#plt.subplot(211)
#plt.title("Accuracy")
#plt.plot(model.history["acc"], color="g", label="train")
#plt.plot(model.history["val_acc"], color="b", label="validation")
#plt.legend(loc="best")

#plt.subplot(212)
#plt.title("Loss")
#plt.plot(model.history["loss"], color="g", label="train")
#plt.plot(model.history["val_loss"], color="b", label="validation")
#plt.legend(loc="best")

#plt.tight_layout()
#plt.show()

# labels
#ytest = np.argmax(Ytest1, axis=1)

# get predictions
Ytest_ = model.predict([Xstest1, Xqtest1])
#ytest_ = np.argmax(Ytest_, axis=1)

NameError: name 'model' is not defined